In [1]:
import os
import sys

In [2]:
sys.path.append(os.path.abspath("../.."))

In [3]:
from datalemur_utils import extract_iframe_from_url

In [4]:
url = "https://datalemur.com/questions/rolling-average-tweets"

In [5]:
extract_iframe_from_url(url)

In [6]:
from pyspark.sql import SparkSession

In [7]:
spark = SparkSession.builder.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/07 02:15:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [8]:
tweets = spark.read\
.option("delimiter", "\t")\
.option("header", "true")\
.option("inferSchema", "true")\
.format("csv")\
.load("tweets.txt")

In [9]:
tweets.createOrReplaceTempView("tweets")

In [10]:
tweets.show()

+-------+-------------------+-----------+
|user_id|         tweet_date|tweet_count|
+-------+-------------------+-----------+
|    111|06/01/2022 00:00:00|          2|
|    111|06/02/2022 00:00:00|          1|
|    111|06/03/2022 00:00:00|          3|
|    111|06/04/2022 00:00:00|          4|
|    111|06/05/2022 00:00:00|          5|
+-------+-------------------+-----------+



In [15]:
spark.sql(
    """
    select 
    user_id, tweet_date,
    round(avg(tweet_count) over(partition by user_id rows between 2 preceding and current row), 2) as rolling_avg_3d
    from tweets
    """
).show()

+-------+-------------------+--------------+
|user_id|         tweet_date|rolling_avg_3d|
+-------+-------------------+--------------+
|    111|06/01/2022 00:00:00|           2.0|
|    111|06/02/2022 00:00:00|           1.5|
|    111|06/03/2022 00:00:00|           2.0|
|    111|06/04/2022 00:00:00|          2.67|
|    111|06/05/2022 00:00:00|           4.0|
+-------+-------------------+--------------+



In [16]:
spark.stop()